código feito pelo apresentador da aula.

Inicialmente, são feitas as importações necessárias para o funcionamento das validações. destaque para a importação da biblioteca pydantic, que é o que de fato iremos usar para manipular as validações de informações, as outras são auxiliares.

In [1]:
from enum import auto, IntFlag
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    SecretStr,
    ValidationError,
)


Duas classes são definidas herdando das importações feitas. A classe Role herda IntFlag, que indica que Role pode receber um dos valores mostrados na classe, sendo o admin a combinação de todos. A classe User herda de BaseModel, que garante que campos obrigatórios padrões (name, email, password, role) estejam presentes na classe e usando o formato correto (ex: email precisa ter @, nomes não podem ter números e etc). O Field serve para adicionar informações extras no campo em questão;.

In [9]:
class Role(IntFlag):
    Author = auto()
    Editor = auto()
    Developer = auto()
    Admin = Author | Editor | Developer


class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["example@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )
    role: Role = Field(default=None, description="The role of the user")

A função validate recebe parâmetro do tipo dicionário e tenta transformá-lo em uma instância da classe User, com o model_validate (método herdado da classe BaseModel) que verifica se os dados obrigatórios existem, validando também a forma como os dados foram redigidos. Caso esteja tudo correto, printa o objeto user gerado a partir da transformação. Caso contrário, printa mensagem de usuário inválido e mostra qual foi o erro detectado.

In [10]:
def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid")
        for error in e.errors():
            print(error)

Na função main abaixo mostra os testes feitos. Temos o dicionário good_data, que possui os dados estruturados da forma correta, e o bad_data, que possui dados incorretos. Ao executar, vemos que a primeira chamada da função "validate" retorna os dados do usuário, e a segunda retorna erro, indicando a falta do campo "name" e o formato incorreto do email.

In [11]:
def main() -> None:
    good_data = {
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
    }
    bad_data = {"email": "<bad data>", "password": "<bad data>"}

    validate(good_data)
    validate(bad_data)

if __name__ == "__main__":
    main()

name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=None
User is invalid
{'type': 'missing', 'loc': ('name',), 'msg': 'Field required', 'input': {'email': '<bad data>', 'password': '<bad data>'}, 'url': 'https://errors.pydantic.dev/2.12/v/missing'}
{'type': 'value_error', 'loc': ('email',), 'msg': 'value is not a valid email address: An email address must have an @-sign.', 'input': '<bad data>', 'ctx': {'reason': 'An email address must have an @-sign.'}}
